# Expanded release EDA (unexecuted)

**Status:** saved with empty outputs and null execution counts. Nothing in this notebook has been executed or verified by Claude Code.

**Purpose:** descriptive inspection of the expanded release derived files. **No model is trained here.**

**Inputs (generated by you, see `EXPANDED_DATASET_RUNBOOK.md`):**
* the expanded manifest (stage 2) and the derived folder `QCP_DATA_ROOT/derived/expanded/<manifest>/expanded-label-v1/<inputs>/` (stage 3 or 6)
* optionally `reports/expanded/validation_full_<manifest>.json` (stage 6)

**Rules:**
* Run the cells one at a time, top to bottom. A missing input raises `MissingEvidenceError` naming the step; there is no demonstration data.
* A manifest entry is a *proposed* key. Only `complete` and `right_censored` keys are usable labels.
* Factor effects are shown as matched-pair ratios (same query instance, or same group/regime), never as pooled comparisons.
* Planner costs are planner units, not milliseconds: compare by rank only.

In [ ]:
# 0.1 Setup: load .env, locate the current expanded manifest and its latest derived folder.
%matplotlib inline
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from query_cost_predictor.envfile import load_repo_env
from query_cost_predictor.errors import MissingEvidenceError
from query_cost_predictor.expanded_derive import latest_expanded_derived_dir
from query_cost_predictor.expanded_manifest import KIND, report_path
from query_cost_predictor.metrics import rank_correlations
from query_cost_predictor.smoke_manifest import ManifestError, current_manifest_id, load_manifest
from query_cost_predictor.splits import assert_groups_disjoint

load_repo_env()
try:
    manifest_id = current_manifest_id(None, KIND)
    manifest, manifest_path = load_manifest(manifest_id, KIND)
except ManifestError as exc:
    raise MissingEvidenceError(f"no expanded manifest ({exc}); run 08_expanded_release.ps1 -Step Manifest") from exc
derived = latest_expanded_derived_dir(manifest_id)  # MissingEvidenceError names the Derive step
print(manifest_id)
print(derived)

In [ ]:
# 0.2 Load derived tables and reconcile them with the manifest.
labels = pd.read_csv(derived / "labels.csv")
keys = pd.read_csv(derived / "keys.csv")
executions = pd.read_csv(derived / "executions.csv")
estimates = pd.read_csv(derived / "estimates.csv")
splits = pd.read_csv(derived / "splits.csv")
derived_manifest = json.loads((derived / "derived_manifest.json").read_text(encoding="utf-8"))
reconciliation = pd.Series({
    "manifest entries": len(manifest["entries"]),
    "label rows": len(labels),
    "key rows": len(keys),
    "split rows": len(splits),
    "usable keys (complete + right_censored)": int(labels["status"].isin(["complete", "right_censored"]).sum()),
    "measured executions (children, not keys)": int((executions["run_purpose"] == "measured").sum()),
})
display(reconciliation)
assert len(labels) == len(keys) == len(splits) == len(manifest["entries"]), "derived files do not cover the manifest"

**Checkpoint 1:** label, key and split rows equal the manifest entries. The usable-key count is what may be quoted, and only after a full validation PASS.

In [ ]:
# 1.1 Observation status by component, snapshot and configuration.
status = labels.pivot_table(index=["component", "snapshot_name", "configuration_name"], columns="observation_status",
                            values="modeling_key", aggfunc="count", fill_value=0)
display(status)
display(labels.loc[labels["status"] == "failed", "failure_class"].value_counts().rename("failed keys by class"))

In [ ]:
# 2.1 Runtime distribution per snapshot (complete keys; censored keys drawn at their execution-time bound).
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, snapshot in zip(axes, ["tpch_sf0_1", "tpch_sf1"]):
    subset = labels[labels["snapshot_name"] == snapshot]
    for component, rows in subset[subset["status"] == "complete"].groupby("component"):
        ax.hist(np.log10(rows["median_ms"].clip(lower=0.001)), bins=40, alpha=0.5, label=f"{component} (n={len(rows)})")
    censored = subset[subset["status"] == "right_censored"]
    if len(censored):
        ax.axvline(np.log10(censored["exec_censor_lower_bound_ms"].min()), linestyle="--", color="black",
                   label=f"censored keys (n={len(censored)}), bound")
    for threshold in (1000, 10000):
        ax.axvline(np.log10(threshold), color="grey", linewidth=0.8)
    ax.set_title(snapshot)
    ax.set_xlabel("log10 median measured execution time (ms)")
    ax.legend(fontsize=7)
axes[0].set_ylabel("keys")
plt.tight_layout()
plt.show()

In [ ]:
# 3.1 Factor separation: matched-pair ratios of median runtime (complete keys only).
complete = labels[labels["status"] == "complete"]


def paired(df, vary, low, high, hold):
    wide = df[df[vary].isin([low, high])].pivot_table(index=hold, columns=vary, values="median_ms", aggfunc="first")
    wide = wide.dropna(subset=[low, high])
    return (wide[high] / wide[low]).rename(f"{high} / {low}")


ratios = {
    "work_mem: C2 / C1": paired(complete, "configuration_name", "C1_baseline", "C2_reduced_work_mem",
                                ["query_instance_id", "snapshot_name"]),
    "scale: SF 1 / SF 0.1": paired(complete, "snapshot_name", "tpch_sf0_1", "tpch_sf1",
                                   ["query_instance_id", "configuration_name"]),
    "index: unindexed / indexed": paired(complete, "index_variant", "indexed_predicate", "unindexed_predicate",
                                         ["semantic_group_id", "parameter_regime", "snapshot_name", "configuration_name"]),
}
display(pd.DataFrame({name: {"pairs": len(r), "median ratio": r.median() if len(r) else None,
                             "share > 1": (r > 1).mean() if len(r) else None} for name, r in ratios.items()}).T)
fig, ax = plt.subplots(figsize=(8, 4))
ax.boxplot([np.log10(r) for r in ratios.values() if len(r)], labels=[n for n, r in ratios.items() if len(r)])
ax.axhline(0, color="grey", linewidth=0.8)
ax.set_ylabel("log10 runtime ratio")
plt.show()

In [ ]:
# 3.2 Generated structural groups: median runtime by selectivity regime, index variant and snapshot.
structural = complete[complete["component"] == "structural"]
display(structural.pivot_table(index=["parameter_regime", "index_variant"], columns="snapshot_name",
                               values="median_ms", aggfunc="median"))

In [ ]:
# 4.1 Frozen high-runtime classes (development keys). Too few positives means NOT_YET_EVALUABLE: no classifier.
development = labels[labels["split_role"] == "development"]
for column in ("hr_1s_class", "hr_10s_class"):
    display(development.pivot_table(index="component", columns=column, values="modeling_key", aggfunc="count", fill_value=0))
    positives = development[development[column] == "positive"]
    print(column, "positive keys:", len(positives), "groups:", positives["semantic_group_id"].nunique(),
          "components:", sorted(positives["component"].unique()))
full_report = report_path("validation_full", manifest_id)
if full_report.is_file():
    gates = json.loads(full_report.read_text(encoding="utf-8"))["gates"]
    display(pd.DataFrame(gates)[["id", "gate", "status"]])
else:
    print("No full validation report yet: gates not evaluated (run 08_expanded_release.ps1 -Step Validate).")

In [ ]:
# 5.1 Planner estimated cost (plain EXPLAIN, planner units) versus measured runtime; rank agreement only.
joined = labels.merge(estimates[["modeling_key", "est_total_cost"]], on="modeling_key", how="inner")
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, snapshot in zip(axes, ["tpch_sf0_1", "tpch_sf1"]):
    subset = joined[joined["snapshot_name"] == snapshot]
    done = subset[subset["status"] == "complete"]
    cens = subset[subset["status"] == "right_censored"]
    ax.scatter(done["est_total_cost"], done["median_ms"], s=6, alpha=0.5, label=f"complete (n={len(done)})")
    ax.scatter(cens["est_total_cost"], cens["exec_censor_lower_bound_ms"], s=14, marker="^", color="black",
               label=f"censored at bound (n={len(cens)})")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_title(snapshot)
    ax.set_xlabel("estimated total cost (planner units)")
    ax.legend(fontsize=7)
axes[0].set_ylabel("median execution time (ms)")
plt.show()
rows = []
for (component, snapshot), group in joined[joined["status"] == "complete"].groupby(["component", "snapshot_name"]):
    if len(group) >= 5:
        rows.append({"component": component, "snapshot": snapshot, "complete keys": len(group),
                     "Spearman": rank_correlations(group["est_total_cost"], group["median_ms"])["spearman"]})
display(pd.DataFrame(rows))

In [ ]:
# 6.1 Spill evidence (execution-only diagnostic, class E) by configuration and snapshot.
measured = executions[executions["run_purpose"] == "measured"].merge(
    labels[["modeling_key", "configuration_name", "snapshot_name"]], on="modeling_key")
spill = measured.groupby(["configuration_name", "snapshot_name"])["spill_detected"].apply(
    lambda s: (s.astype(str).str.lower() == "true").mean())
display(spill.rename("share of measured runs with spill"))

In [ ]:
# 7.1 Folds: re-check that no semantic group crosses primary folds and show keys/groups per fold.
dev_splits = splits[splits["split_role"] == "development"]
assert_groups_disjoint(dev_splits["fold_group"].astype(int).tolist(), dev_splits["semantic_group_id"].tolist())
usable = labels[labels["status"].isin(["complete", "right_censored"])][["modeling_key"]]
per_fold = dev_splits.merge(usable, on="modeling_key").groupby("fold_group").agg(
    usable_keys=("modeling_key", "count"), groups=("semantic_group_id", "nunique"))
per_fold["keys_per_group"] = per_fold["usable_keys"] / per_fold["groups"]
display(per_fold)
print("RQ2-reserved keys (never in an RQ1 fold):", int((splits["split_role"] == "heldout_rq2").sum()))

## Limitations

* TPC-H-derived benchmark data at SF 0.1 and SF 1 on one laptop: not a production workload (see `docs/query_workload_evidence_register.md`).
* Instrumented `EXPLAIN ANALYZE` execution time (warm, single client) is not end-to-end latency and not a dollar cost.
* Runs beyond the 15 s timeout are right-censored; their bound is the execution-time bound, not a runtime.
* Semantic groups share some access paths; the stricter `fold_family` split must be reported next to the primary split.
* The 25 manual rewrite pairs are dataset keys whose equivalence is `NOT_YET_VERIFIED`.